# 00 Data Processing


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
from numpy import linalg as la

from config import (
    RAW_MATRICES_DIR, WAVEFUNCTIONS_DIR,
    PROCESSED_DATAFRAMES_DIR
)
from notebook_utils.general import read_npz

### Statevector Dataset

In [2]:
SV_HAMILTONIAN_DIR = RAW_MATRICES_DIR / "SV_hamiltonian"
SV_SPIN_DIR = RAW_MATRICES_DIR / "SV_spin"
PYSCF_CASCI_DIR = WAVEFUNCTIONS_DIR / "pyscf_casci"

def load_sv_hamiltonian_records(path):
    records = []

    for file in sorted(path.rglob("*.npz")):
        row = read_npz(file)
        H = row["H"]
        S = row["S"]

        records.append({
            "basis": row["basis"],
            "molecule": row["molecule"],
            "active_space": row["active_space"],
            "ansatz": row["ansatz"],
            "expansion": row["expansion"],
            "H": H,
            "S": S,
            "qse_dim": H.shape[0],
        })

    return pd.DataFrame(records)


def load_spin_records(path):
    records = []

    for file in sorted(path.rglob("*.npz")):
        row = read_npz(file)
        S2 = row["Z"]

        records.append({
            "basis": row["basis"],
            "molecule": row["molecule"],
            "active_space": row["active_space"],
            "ansatz": row["ansatz"],
            "expansion": row["expansion"],
            "S2": S2,
        })

    return pd.DataFrame(records)

def load_pyscf_casci_energy_records(path):
    records = []

    for file in sorted(path.rglob("*.npz")):
        row = read_npz(file)
        sectors = row["sectors"].tolist()

        if row["spin_type"] == "singlet":
            casci_energies = row["casci_energies"].tolist()
            casci_pvec = {
                str(root): vector
                for root, vector in enumerate(row["casci_pvec"])
            }
        else:
            casci_energies = [
                {sector: float(energy) for sector, energy in zip(sectors, nth_energies)}
                for nth_energies in row["casci_energies"]
            ]
            casci_pvec = {
                f"{root}_{sector}": vector
                for root, root_vectors in enumerate(row["casci_pvec"])
                for sector, vector in zip(sectors, root_vectors)
            }

        records.append({
            "basis": row["basis"],
            "molecule": row["molecule"],
            "active_space": row["active_space"],
            "spin_type": row["spin_type"],
            "pyscf_casci_energies": casci_energies,
            "pyscf_casci_pvec": casci_pvec,
        })

    return pd.DataFrame(records)


df_sv_hamiltonian = load_sv_hamiltonian_records(SV_HAMILTONIAN_DIR)
df_sv_spin = load_spin_records(SV_SPIN_DIR)
df_pyscf_casci = load_pyscf_casci_energy_records(PYSCF_CASCI_DIR)


In [3]:
# Compress the PYSCF Triplet energies
def average_triplet_sector_energies(casci_energies):
    average_energies = []
    sector_diffs = []

    for nth_energies in casci_energies:
        energies = list(nth_energies.values())
        average_energies.append(float(np.mean(energies)))
        sector_diffs.append(float(np.ptp(energies)))

    return average_energies, sector_diffs


new_casci_energies = []
triplet_sector_diffs = []

for _, row in df_pyscf_casci.iterrows():
    if row["spin_type"] == "triplet":
        energies, sector_diffs = average_triplet_sector_energies(
            row["pyscf_casci_energies"]
        )
        triplet_sector_diffs.extend(sector_diffs)
    else:
        energies = row["pyscf_casci_energies"]

    new_casci_energies.append(energies)


df_pyscf_casci["pyscf_casci_energies"] = new_casci_energies

max_triplet_sector_diff = max(triplet_sector_diffs, default=0.0)
print(f"Maximum PYSCF triplet sector energy difference: {max_triplet_sector_diff:.12e} Ha")


Maximum PYSCF triplet sector energy difference: 1.004013938655e-08 Ha


In [4]:
qse_keys = ["basis", "molecule", "active_space", "ansatz", "expansion"]

df_sv = df_sv_hamiltonian.merge(df_sv_spin, on=qse_keys, how="left", validate="one_to_one")

df_sv = df_sv.merge(
    df_pyscf_casci,
    left_on=["basis", "molecule", "active_space", "expansion"],
    right_on=["basis", "molecule", "active_space", "spin_type"],
    how="left", validate="many_to_one",
).drop(columns="spin_type")

df_sv = df_sv.sort_values(qse_keys).reset_index(drop=True)

In [5]:
required_columns = ["H", "S", "pyscf_casci_energies", "pyscf_casci_pvec"]

df_sv_complete = df_sv.dropna(subset=required_columns).reset_index(drop=True)

PROCESSED_DATAFRAMES_DIR.mkdir(parents=True, exist_ok=True)
SV_DATAFRAME_PATH = PROCESSED_DATAFRAMES_DIR / "sv_qse_data.pkl"

df_sv_complete.to_pickle(SV_DATAFRAME_PATH)

df_sv_complete


,basis,molecule,active_space,ansatz,expansion,H,S,qse_dim,S2,pyscf_casci_energies,pyscf_casci_pvec
0,cc-pVDZ,Acetamide,6e6o,UCCSD,singlet,"[[(-829.6506096281651+0j), (-0.000181302476295...","[[(3.988712948673631+0j), (8.726515365415764e-...",36,NaN,"[-208.00133775639495, -207.7187801970414, -207...","{'0': [1.2699592266214175e-06, 0.0, -1.6751938..."
1,cc-pVDZ,Acetamide,6e6o,UCCSD,triplet,"[[(-0.19501209940447028+0j), 0j, 0j, (-3.78300...","[[(0.0009407665029827761+0j), 0j, 0j, (1.82567...",108,NaN,"[-207.73202945843013, -207.71304204527416, -20...","{'0_0': [-8.326672684688674e-17, 0.0, 2.072432..."
2,cc-pVDZ,Acetone,6e6o,UCCSD,singlet,"[[(-767.8077986867438+0j), (4.0815803667927343...","[[(3.9992858479784488+0j), (-2.137399598149104...",36,NaN,"[-191.98641789032493, -191.7602163048087, -191...","{'0': [1.9066199583753638e-06, 0.0, -1.3241602..."
3,cc-pVDZ,Acetone,6e6o,UCCSD,triplet,"[[(-0.015424107753749753+0j), 0j, 0j, (6.68623...","[[(8.076072143800572e-05+0j), 0j, 0j, (-3.5042...",108,NaN,"[-191.7770411769644, -191.71265341094087, -191...","{'0_0': [-6.938893903907228e-17, 0.0, -1.14929..."
4,cc-pVDZ,Adenine,6e6o,UCCSD,singlet,"[[(-1858.1495721293768+0j), (-1.14829265075595...","[[(3.999674884288658+0j), (2.4718027004076955e...",36,NaN,"[-464.5752925239075, -464.32010428779466, -464...","{'0': [1.2183945000148156e-05, 0.0, 1.24502022..."
...,...,...,...,...,...,...,...,...,...,...,...
2571,sto-3g,Uracil,6e7o,UCCSD,triplet,"[[(-0.11042245041486087+0j), 0j, 0j, (-6.37046...","[[(0.0002718226282586167+0j), 0j, 0j, (1.56773...",147,"[[(0.0005436414375037656+0j), 0j, 0j, (3.13543...","[-406.95689685600695, -406.90419213472666, -40...","{'0_0': [-3.919148314149759e-08, 0.0, -3.51006..."
2572,sto-3g,Uracil,8e7o,UCCSD,singlet,"[[(-1627.9179451613252+0j), (0.151178922494367...","[[(3.998609492626249+0j), (-0.0003714519947059...",49,"[[(7.043982743019104e-07+0j), (3.4870941333265...","[-407.1214804205012, -406.94605481069067, -406...","{'0': [-2.3296044344376372e-06, 0.0, -8.024116..."
2573,sto-3g,Uracil,8e7o,UCCSD,triplet,"[[(-0.05217935944762317+0j), 0j, 0j, (-0.01252...","[[(0.00012840277216497054+0j), 0j, 0j, (3.0821...",147,"[[(0.0002567898781715119+0j), 0j, 0j, (6.16475...","[-406.96118398246017, -406.95666451956214, -40...","{'0_0': [-1.949602093463748e-15, 0.0, -4.70314..."
2574,sto-3g,Uracil,8e8o,UCCSD,singlet,"[[(-1627.5906696076613+0j), (0.352919530145391...","[[(3.997797662312483+0j), (-0.0008670055522641...",64,"[[(1.4686254965902396e-06+0j), (-4.62676091070...","[-407.12243001960377, -406.9465993150982, -406...","{'0': [-1.2617166801127624e-06, 0.0, -1.826998..."


### OV-doubles statevector dataset

In [6]:
SV_OV_DOUBLES_DIR = RAW_MATRICES_DIR / "SV_hamiltonian_select_doubles" / "ov_doubles"

df_sv_ov_doubles_hamiltonian = load_sv_hamiltonian_records(SV_OV_DOUBLES_DIR)
df_singlet_casci = df_pyscf_casci[df_pyscf_casci["spin_type"] == "singlet"].drop(columns="spin_type")

df_sv_ov_doubles = df_sv_ov_doubles_hamiltonian.merge(
    df_singlet_casci, on=["basis", "molecule", "active_space"], how="left", validate="many_to_one"
).sort_values(qse_keys).reset_index(drop=True)

required_columns = ["H", "S", "pyscf_casci_energies", "pyscf_casci_pvec"]
df_sv_ov_doubles = df_sv_ov_doubles.dropna(subset=required_columns).reset_index(drop=True)

OV_DOUBLES_DATAFRAME_PATH = PROCESSED_DATAFRAMES_DIR / "sv_qse_ov_doubles_data.pkl"
df_sv_ov_doubles.to_pickle(OV_DOUBLES_DATAFRAME_PATH)

df_sv_ov_doubles


,basis,molecule,active_space,ansatz,expansion,H,S,qse_dim,pyscf_casci_energies,pyscf_casci_pvec
0,cc-pVDZ,Acetamide,6e6o,UCCSD,select_ov_doubles,"[[(-414.82530481408253+0j), (-9.06512381475183...","[[(1.9943564743368152+0j), (4.36325768270788e-...",81,"[-208.00133775639495, -207.7187801970414, -207...","{'0': [1.2699592266214175e-06, 0.0, -1.6751938..."
1,cc-pVDZ,Acetone,6e6o,UCCSD,select_ov_doubles,"[[(-383.9038993433718+0j), (2.040790183396367e...","[[(1.999642923989224+0j), (-1.068699799074552e...",81,"[-191.98641789032493, -191.7602163048087, -191...","{'0': [1.9066199583753638e-06, 0.0, -1.3241602..."
2,cc-pVDZ,Adenine,6e6o,UCCSD,select_ov_doubles,"[[(-929.0747860646884+0j), (-5.741463253779788...","[[(1.9998374421443286+0j), (1.2359013502038474...",81,"[-464.5752925239075, -464.32010428779466, -464...","{'0': [1.2183945000148156e-05, 0.0, 1.24502022..."
3,cc-pVDZ,Benzene,6e6o,UCCSD,select_ov_doubles,"[[(-461.47605641220014+0j), (-7.58860797451898...","[[(1.999844848583004+0j), (3.288573797313858e-...",81,"[-230.75601895849644, -230.49368590989286, -23...","{'0': [0.00016739696430541302, 0.0, -7.3215656..."
4,cc-pVDZ,Benzoquinone,6e6o,UCCSD,select_ov_doubles,"[[(-758.3901420553599+0j), (-1.915178924100565...","[[(1.99951875007491+0j), (5.049564607353619e-1...",81,"[-379.2864476632125, -379.0736566181682, -379....","{'0': [0.00010945052129107436, 0.0, -9.2088400..."
5,cc-pVDZ,Butadiene,6e6o,UCCSD,select_ov_doubles,"[[(-309.80382193167463+0j), (3.329183134475539...","[[(1.9994051270563677+0j), (-2.149392213102799...",81,"[-154.9482614423485, -154.66564617914867, -154...","{'0': [7.747540036845018e-06, 0.0, 1.455033670..."
6,cc-pVDZ,Cyclopentadiene,6e6o,UCCSD,select_ov_doubles,"[[(-385.5937484313247+0j), (3.2053026112365105...","[[(1.9997725135453057+0j), (-1.644410040524736...",81,"[-192.81889092701402, -192.57119534882662, -19...","{'0': [3.3683314662747534e-06, 0.0, 6.76421489..."
7,cc-pVDZ,Cyclopropene,6e6o,UCCSD,select_ov_doubles,"[[(-231.66835493008406+0j), (-8.24908721031258...","[[(1.9998274139402104+0j), (1.6802757032005795...",81,"[-115.84427180142049, -115.55141172502775, -11...","{'0': [7.077615271029555e-06, 0.0, 3.716078358..."
8,cc-pVDZ,Cytosine,6e6o,UCCSD,select_ov_doubles,"[[(-785.2419732421588+0j), (-2.478007090026067...","[[(1.9998285636985231+0j), (6.31815920942432e-...",81,"[-392.65468007184893, -392.39546004938785, -39...","{'0': [5.414086936424542e-07, 0.0, -5.41799256..."
9,cc-pVDZ,Ethene,6e6o,UCCSD,select_ov_doubles,"[[(-156.06982796125814+0j), (-4.58852421937844...","[[(1.9993922480465662+0j), (5.8861188955023494...",81,"[-78.05899287410917, -77.70313410502304, -77.6...","{'0': [3.651300513430399e-05, 0.0, 5.525845344..."


### Shots dataset

In [7]:
SHOTS_HAMILTONIAN_DIR = RAW_MATRICES_DIR / "shots_hamiltonian"


def load_shots_hamiltonian_records(path):
    records = []

    for file in sorted(path.rglob("*.npz")):
        row = read_npz(file)
        h_keys = sorted(key for key in row if key.startswith("H_shots_"))

        for h_key in h_keys:
            sample_key = h_key.removeprefix("H_")
            _, n_shots, repeat = sample_key.split("_")
            H = row[f"H_{sample_key}"]
            S = row[f"S_{sample_key}"]

            records.append({
                "basis": row["basis"],
                "molecule": row["molecule"],
                "active_space": row["active_space"],
                "ansatz": row["ansatz"],
                "expansion": row["expansion"],
                "sample_key": sample_key,
                "n_shots": int(n_shots),
                "repeat": int(repeat),
                "H_shots": H,
                "S_shots": S,
                "qse_dim": H.shape[0],
                "num_commuting_groups": int(row["num_commuting_groups"]),
            })

    return pd.DataFrame(records)


df_shots_hamiltonian = load_shots_hamiltonian_records(SHOTS_HAMILTONIAN_DIR)

df_shots_sv = df_sv_hamiltonian.rename(columns={
    "H": "H_sv",
    "S": "S_sv",
    "qse_dim": "qse_dim_sv",
})

shot_keys = qse_keys + ["n_shots", "repeat"]
assert not df_shots_hamiltonian.duplicated(shot_keys).any(), "Duplicate shot/repeat records"

df_shots = df_shots_hamiltonian.merge(df_shots_sv, on=qse_keys, how="left", validate="many_to_one")
df_shots = df_shots.sort_values(shot_keys).reset_index(drop=True)


In [8]:
required_columns = ["H_shots", "S_shots", "H_sv", "S_sv"]

df_shots_complete = df_shots.dropna(subset=required_columns).reset_index(drop=True)

PROCESSED_DATAFRAMES_DIR.mkdir(parents=True, exist_ok=True)
SHOTS_DATAFRAME_PATH = PROCESSED_DATAFRAMES_DIR / "shots_qse_data.pkl"

df_shots_complete.to_pickle(SHOTS_DATAFRAME_PATH)

df_shots_complete


,basis,molecule,active_space,ansatz,expansion,sample_key,n_shots,repeat,H_shots,S_shots,qse_dim,num_commuting_groups,H_sv,S_sv,qse_dim_sv
0,sto-3g,Acetamide,2e2o,UCCSD,singlet,shots_1000_0,1000,0,"[[(-799.3416862999425+0j), (-9.698487335464566...","[[(3.894+0j), (0.047250000000000014+0.02125000...",4,25,"[[(-802.566868206377+0j), (-4.71630877716983+0...","[[(3.9097067555916256+0j), (0.0229713600038418...",4
1,sto-3g,Acetamide,2e2o,UCCSD,singlet,shots_1000_1,1000,1,"[[(-798.521323691679+0j), (-5.749462200016003+...","[[(3.8899999999999997+0j), (0.0280000000000000...",4,25,"[[(-802.566868206377+0j), (-4.71630877716983+0...","[[(3.9097067555916256+0j), (0.0229713600038418...",4
2,sto-3g,Acetamide,2e2o,UCCSD,singlet,shots_1000_2,1000,2,"[[(-799.7532554896775+0j), (-5.648868316573327...","[[(3.896+0j), (0.027499999999999997-0.00400000...",4,25,"[[(-802.566868206377+0j), (-4.71630877716983+0...","[[(3.9097067555916256+0j), (0.0229713600038418...",4
3,sto-3g,Acetamide,2e2o,UCCSD,singlet,shots_1000_3,1000,3,"[[(-804.2721954287647+0j), (-7.493082463093295...","[[(3.918+0j), (0.03650000000000003+0.021000000...",4,25,"[[(-802.566868206377+0j), (-4.71630877716983+0...","[[(3.9097067555916256+0j), (0.0229713600038418...",4
4,sto-3g,Acetamide,2e2o,UCCSD,singlet,shots_1000_4,1000,4,"[[(-799.3431587075672+0j), (3.7451390568854235...","[[(3.894+0j), (-0.01825000000000003-0.03225000...",4,25,"[[(-802.566868206377+0j), (-4.71630877716983+0...","[[(3.9097067555916256+0j), (0.0229713600038418...",4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44795,sto-3g,Uracil,8e7o,UCCSD,singlet,shots_1000000_5,1000000,5,"[[(-1627.8978781435267+0j), (-0.20464641469821...","[[(3.998559+0j), (0.0005032500000000453+0.0011...",49,142140,"[[(-1627.9179451613252+0j), (0.151178922494367...","[[(3.998609492626249+0j), (-0.0003714519947059...",49
44796,sto-3g,Uracil,8e7o,UCCSD,singlet,shots_1000000_6,1000000,6,"[[(-1627.9323084357866+0j), (0.089104040640988...","[[(3.9986420000000003+0j), (-0.000219499999999...",49,142140,"[[(-1627.9179451613252+0j), (0.151178922494367...","[[(3.998609492626249+0j), (-0.0003714519947059...",49
44797,sto-3g,Uracil,8e7o,UCCSD,singlet,shots_1000000_7,1000000,7,"[[(-1627.923975251706+0j), (0.5163541613184499...","[[(3.9986260000000002+0j), (-0.001272250000000...",49,142140,"[[(-1627.9179451613252+0j), (0.151178922494367...","[[(3.998609492626249+0j), (-0.0003714519947059...",49
44798,sto-3g,Uracil,8e7o,UCCSD,singlet,shots_1000000_8,1000000,8,"[[(-1627.9118992952774+0j), (0.308116564461623...","[[(3.998592+0j), (-0.0007580000000000104-0.000...",49,142140,"[[(-1627.9179451613252+0j), (0.151178922494367...","[[(3.998609492626249+0j), (-0.0003714519947059...",49
